In [67]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import json
import os
import sys

from preprocessing.ecg_preprocessor import ECGPreprocessor
from preprocessing.lead_processor import LeadProcessor

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [81]:
test_ecg_path = os.path.expanduser("~/UNI/CORSI/Visione_Artificiale/progetto/datasets/PTB/s0001_re.csv")
lead_names = ["i","ii","iii","avl","avr","avf","v1","v2","v3","v4","v5","v6"]
df = pd.read_csv(test_ecg_path)[lead_names]
data_in = df.to_numpy().T
freq_hz = 500

preprocessor = ECGPreprocessor(freq_hz)
results_post_filters, m_detrend, error_preprocessing = preprocessor.run(data_in)

for lead_to_test in range(12):
    print(f"TESTING LEAD {lead_to_test} ----------------------------------------------------")
    lead_i = LeadProcessor(data_in[lead_to_test], lead_idx=lead_to_test, freq_hz=freq_hz)
    lead_i.preprocess_single_lead()
    pre_I = lead_i.to_dict()

    path = f"/home/giava/UNI/CORSI/Visione_Artificiale/progetto/FMM-Applications-3DECG/test_porting/lead_{lead_to_test+1}"
    with open(f"{path}/R_test_pre_I.json", "r") as f: r_pre_I = json.load(f)

    # TEST LOESSDATA (segnale ECG con rimozione baseline wander)
    try:
        np.testing.assert_allclose(pre_I["loessData"], np.array(r_pre_I["loessData"]), rtol=1e-5, atol=1e-5)
        print("✅ loessData matches!")
    except AssertionError as e:
        print("❌ loessData DOES NOT MATCH!")
        print(e)

    # TEST PICCHI R
    try:
        np.testing.assert_allclose(pre_I["loessRPeaksEnd"], np.array(r_pre_I["loessRPeaksEnd"]) - 1, rtol=1e-5, atol=1e-5)
        print("✅ loessRPeaksEnd matches!")
    except AssertionError as e:
        print("❌ loessRPeaksEnd DOES NOT MATCH!")
        print(e)

    # TEST SEGMENTI QRS
    try:
        np.testing.assert_allclose(pre_I["loessSegEnd"], np.array(r_pre_I["loessSegEnd"]) - 1, rtol=1e-5, atol=1e-5)
        print("✅ loessSegEnd matches!")
    except AssertionError as e:
        print("❌ loessSegEnd DOES NOT MATCH!")
        print(e)

# TEST m_detrend (segnale pulito) (matrice con tutte le derivazioni)
print("TEST detrended signal -----------------------------------------")
with open(f"{path}/R_test_m_detrend.json", "r") as f: r_m_detrend = json.load(f)

try:
    np.testing.assert_allclose(m_detrend, np.array(r_m_detrend).T, rtol=1e-5, atol=1e-5)
    print("✅ m_detrend matches!")
except AssertionError as e:
    print("❌ m_detrend DOES NOT MATCH!")
    print(e)


TESTING LEAD 0 ----------------------------------------------------
✅ loessData matches!
✅ loessRPeaksEnd matches!
✅ loessSegEnd matches!
TESTING LEAD 1 ----------------------------------------------------
✅ loessData matches!
✅ loessRPeaksEnd matches!
✅ loessSegEnd matches!
TESTING LEAD 2 ----------------------------------------------------
✅ loessData matches!
✅ loessRPeaksEnd matches!
✅ loessSegEnd matches!
TESTING LEAD 3 ----------------------------------------------------
✅ loessData matches!
✅ loessRPeaksEnd matches!
✅ loessSegEnd matches!
TESTING LEAD 4 ----------------------------------------------------
✅ loessData matches!
✅ loessRPeaksEnd matches!
✅ loessSegEnd matches!
TESTING LEAD 5 ----------------------------------------------------
✅ loessData matches!
✅ loessRPeaksEnd matches!
✅ loessSegEnd matches!
TESTING LEAD 6 ----------------------------------------------------
✅ loessData matches!
✅ loessRPeaksEnd matches!
✅ loessSegEnd matches!
TESTING LEAD 7 -------------------

In [82]:
final_output_R = []

with open(f"{path}/R_test_pos_results.json", "r") as f: r_pos_results = json.load(f)


for beat in r_pos_results:
    # | QRS location | beat start | beat end |
    final_output_R.append(beat[1:4])

der = 11
beats_der = results_post_filters[der]['loessRPeaksEnd']
start_end_der = results_post_filters[der]['loessSegEnd']

final_output_python = []

for i in range(len(beats_der)):
    final_output_python.append([int(beats_der[i])+1, int(start_end_der[i][0])+1, int(start_end_der[i][1])+1])

In [83]:
try:
    np.testing.assert_allclose(final_output_python, final_output_R, rtol=1e-5, atol=1e-5)
    print("✅ pos_results matches!")
except AssertionError as e:
    print("❌ pos_results DOES NOT MATCH!")

print(final_output_R)
print(final_output_python)


❌ pos_results DOES NOT MATCH!
[[303, 1, 798], [1128, 799, 1605], [1923, 1606, 2406], [2728, 2407, 3216], [3541, 3217, 4056], [4398, 4057, 4908], [5247, 4909, 5738], [6065, 5739, 6580], [6922, 6581, 7436], [7778, 7437, 8273], [8603, 8274, 9106], [9441, 9107, 9941], [10273, 9942, 10777], [11112, 10778, 11601], [11926, 11602, 12425], [12757, 12426, 13252], [13582, 13253, 14076], [14405, 14077, 14892], [15216, 14893, 15712], [16042, 15713, 16537], [16866, 16538, 17362], [17692, 17363, 18180], [18505, 18181, 19004], [19336, 19005, 19837], [20170, 19838, 20673], [21008, 20674, 21506], [21837, 21507, 22343], [22680, 22344, 23174], [23502, 23175, 23994], [24321, 23995, 24815], [25143, 24816, 25644], [25978, 25645, 26469], [26796, 26470, 27282], [27606, 27283, 28097], [28423, 28098, 28926], [29261, 28927, 29761], [30094, 29762, 30580], [30904, 30581, 31391], [31715, 31392, 32222], [32559, 32223, 33062], [33397, 33063, 33883], [34207, 33884, 34706], [35038, 34707, 35535], [35866, 35536, 36361], 